In [1]:
import config
import tensorflow.keras as keras
import tensorflow as tf
from CustomLossFunction import dice_coef
from Generator import DataGenerator
import Generator
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ModelBuilding import CreateModel
os.getcwd()

'/shared/WorkAttachment/codes/code6'

In [2]:
try:
    # Disable all GPUS
    tf.config.set_visible_devices([], 'GPU')
    visible_devices = tf.config.get_visible_devices()
    for device in visible_devices:
        assert device.device_type != 'GPU'
except:
    # Invalid device or cannot modify virtual devices once initialized.
    pass

In [3]:
model = keras.models.load_model(os.path.sep.join(["output","models", "20201220", "latest_model"]), custom_objects={'dice_coef': dice_coef})#, compile=True)

ResourceExhaustedError: OOM when allocating tensor with shape[2097152,512] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator mklcpu [Op:Add]

In [ ]:
model = CreateModel()
losses = {
        "class_label": tf.keras.losses.categorical_crossentropy,
        "bounding_box": dice_coef,
    }
# model.load_weights(os.path.sep.join(["output","models","bests", "model.03-1.17.h5"]))
model.load_weights(os.path.sep.join(["output","models","bests", "model.03-0.93.h5"]))
model.compile(loss=losses)

In [ ]:
datagen = DataGenerator(['kidneyHU.png', 'lungHU.png', 'liverHU.png', 'abdomenHU.png', 'boneHU.png', 'mediastinumHU.png', 'pelvisHU.png', 'softtissue.png'], '.', to_fit=False, csvpath=config.ANNOTS_PATH)

In [ ]:
imgarrs = [Generator.ImagePreprocessing(i, (512,512)) for i in ['kidneyHU.png', 'lungHU.png', 'liverHU.png', 'abdomenHU.png', 'boneHU.png', 'mediastinumHU.png', 'pelvisHU.png', 'softtissue.png']]
imgarrs = np.array(imgarrs)
print(imgarrs.shape)

In [ ]:
output = model(imgarrs, training=False)

In [ ]:
for i in range(len(output)):
    print(output[i].shape)

In [ ]:
labels = ['kidneyHU.png', 'lungHU.png', 'liverHU.png', 'abdomenHU.png', 'boneHU.png', 'mediastinumHU.png', 'pelvisHU.png', 'softtissue.png']
for i in range(8):
    leison_types = output[0][i]
    aimg = output[1][i]
#     break
    # print(leison_types)
    print('image', i)
    num = np.argmax(leison_types)
    print('expected\t', labels[i] )
    print('results\t\t', num+1, config.NUM_TYPE_MAPPING[str(num+1)])
    print('='*30)
    # num = leison_types > 0.6
    # for index, j in enumerate(num):
    #     # print(j)
    #     if j:
    #         print(index+1, config.NUM_TYPE_MAPPING[str(index+1)], '\t', leison_types[index])

In [ ]:
# mask = generator.ImagePreprocessing('kidneyHU.png', (512,512))
mask = datagen._generate_y(['000010_01_01_084.png'])['bounding_box'] #kidney
# mask = datagen._generate_y(['000004_02_02_073.png'])['bounding_box'] #lung
# mask = datagen._generate_y(['000016_01_01_008.png'])['bounding_box'] #liver
# mask = datagen._generate_y(['000002_01_01_162.png'])['bounding_box'] #abdomen
# mask = datagen._generate_y(['000010_02_02_100.png'])['bounding_box'] #bone
# mask = datagen._generate_y(['000001_01_01_109.png'])['bounding_box'] #mediastinum
# mask = datagen._generate_y(['000014_02_01_101.png'])['bounding_box'] #pelvis
# mask = datagen._generate_y(['000004_01_01_007.png'])['bounding_box'] #soft tissue
# mask = np.delete(mask, np.s_[1:], axis=-1).reshape(512,512)
# print(mask)
original = plt.imshow(mask.reshape(512,512))
plt.colorbar()

In [ ]:
l = []
for i in range(8):
    numpymask = np.array(output[1][i]).squeeze()
    numpymask = numpymask > 0.5

    l.append(numpymask)
    # plt.hist(numpymask.ravel(), bins=256, range=(0.0, 1.0), fc='k', ec='k')
    imgplot = plt.imshow(numpymask)
    # plt.colorbar()

In [ ]:
# contours,_ = cv2.findContours(numpymask.copy(), 1, 1) # not copying here will throw an error
# rect = cv2.minAreaRect(contours[0]) # basically you can feed this rect into your classifier
# (x,y),(w,h), a = rect # a - angle

# box = cv2.boxPoints(rect)
# box = np.int0(box) #turn into ints
# rect2 = cv2.drawContours(img.copy(),[box],0,(0,0,255),10)

# plt.imshow(rect2)
# plt.show()

In [ ]:
import cv2
from Generator import GenBboxCoord
import config
import os
import numpy as np
fns = ['000010_01_01_084.png', '000004_02_02_073.png', '000016_01_01_008.png', '000002_01_01_162.png', '000010_02_02_100.png', '000001_01_01_109.png', '000014_02_01_101.png', '000004_01_01_007.png']
coords = GenBboxCoord(config.ANNOTS_PATH, fns)
for i, fn_and_coords in enumerate(zip(fns, coords)):
    mask = np.asarray(l[i], np.uint8)
    fn, coords = fn_and_coords
    print(fn)
    img = cv2.imread(os.path.sep.join([config.IMAGES_PATH, fn]))
    # print(type(img[0][0][0]))
    mask.resize((512,512,1), refcheck=False)
    mask = np.repeat(mask, 3, axis=2)
    mask[:,:,:2] = 0
    mask *= 255
    print(max(mask.flatten()))
    alpha = 0.3
    img = cv2.addWeighted(np.asarray(mask, np.uint8), alpha, img, 1-alpha,0)

    for coord in coords:
        offset = 2
        print(coord)
        pt1 = (int(coord[0] + offset), int(coord[1] + offset))
        pt2 = (int(coord[2] + offset), int(coord[3] + offset))
        img = cv2.rectangle(img, pt1, pt2, (255,0,0))
    
    cv2.imshow(str(i), img)
    cv2.waitKey(5000)
    
input('wait')
cv2.destroyAllWindows()
    # del img